# Lab 7 — Navigation with move_base

<svg width="100%" viewBox="0 0 1260 150" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Navigation stack">
<defs><marker id="arrow" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto" markerUnits="strokeWidth"><path d="M0,0 L0,6 L9,3 z" fill="#334155"/></marker></defs>
<rect x="0" y="0" width="1260" height="150" rx="18" fill="#f8fafc" stroke="#cbd5e1"/>
<text x="24" y="30" font-family="Arial" font-size="20" font-weight="700" fill="#0f172a">Navigation stack</text>
<rect x="25.0" y="55" width="180.8" height="58" rx="12" fill="white" stroke="#64748b"/>
<text x="115.4" y="80" font-family="Arial" font-size="14" text-anchor="middle" fill="#0f172a">Saved map</text>
<line x1="205.8" y1="84" x2="224.8" y2="84" stroke="#334155" stroke-width="2" marker-end="url(#arrow)"/>
<rect x="230.8" y="55" width="180.8" height="58" rx="12" fill="white" stroke="#64748b"/>
<text x="321.2" y="80" font-family="Arial" font-size="14" text-anchor="middle" fill="#0f172a">Localization</text>
<line x1="411.7" y1="84" x2="430.7" y2="84" stroke="#334155" stroke-width="2" marker-end="url(#arrow)"/>
<rect x="436.7" y="55" width="180.8" height="58" rx="12" fill="white" stroke="#64748b"/>
<text x="527.1" y="80" font-family="Arial" font-size="14" text-anchor="middle" fill="#0f172a">Global planner</text>
<line x1="617.5" y1="84" x2="636.5" y2="84" stroke="#334155" stroke-width="2" marker-end="url(#arrow)"/>
<rect x="642.5" y="55" width="180.8" height="58" rx="12" fill="white" stroke="#64748b"/>
<text x="732.9" y="80" font-family="Arial" font-size="14" text-anchor="middle" fill="#0f172a">Local planner</text>
<line x1="823.3" y1="84" x2="842.3" y2="84" stroke="#334155" stroke-width="2" marker-end="url(#arrow)"/>
<rect x="848.3" y="55" width="180.8" height="58" rx="12" fill="white" stroke="#64748b"/>
<text x="938.8" y="80" font-family="Arial" font-size="14" text-anchor="middle" fill="#0f172a">Velocity commands</text>
<line x1="1029.2" y1="84" x2="1048.2" y2="84" stroke="#334155" stroke-width="2" marker-end="url(#arrow)"/>
<rect x="1054.2" y="55" width="180.8" height="58" rx="12" fill="white" stroke="#64748b"/>
<text x="1144.6" y="80" font-family="Arial" font-size="14" text-anchor="middle" fill="#0f172a">Robot motion</text>
</svg>


## Learning outcomes
You will load a saved map, initialize localization, send navigation goals in RViz, and write a waypoint script that publishes goal poses.

## Concept snapshot
The `move_base` navigation stack combines localization, costmaps, global planning, local planning, and recovery behaviors. The robot needs a map, an initial pose estimate, sensor updates, and safe controller limits before it can navigate reliably.


## Prepare the map
Copy a saved map pair into the map directory:
```bash
~/jetauto_ws/src/jetauto_slam/maps/
```
The pair should include:
- `room_map.pgm` or your map image
- `room_map.yaml` or matching metadata file


## Start the navigation stack
On the robot:
```bash
sudo systemctl stop start_app_node.service
roslaunch jetauto_controller jetauto_controller.launch
roslaunch jetauto_navigation navigation.launch map:=room_map
```
In another terminal:
```bash
roslaunch jetauto_navigation rviz_navigation.launch
```


## RViz workflow
1. Use **2D Pose Estimate** to align the robot with its real position on the map.
2. Use **2D Nav Goal** to send a goal pose.
3. Watch the global path and local costmap update.
4. If localization is poor, manually drive the robot slowly and reset the initial pose.
5. Test obstacle response only after the robot can navigate reliably without obstacles.


## Waypoint navigation starter script
Create `waypoint_navigation.py` in the `scripts/` folder of your own package. Confirm the goal topic with `rostopic list`; some setups use `/move_base_simple/goal`, while namespaced setups use `/jetauto_1/move_base_simple/goal`.
```python

#!/usr/bin/env python3
import math
import rospy
from geometry_msgs.msg import PoseStamped
from tf.transformations import quaternion_from_euler

WAYPOINTS = [
    (-0.8, 0.2, 0.0),
    (0.5, 1.0, 90.0),
    (1.4, 0.0, 180.0),
]


def make_goal(x, y, yaw_deg, frame_id='map'):
    goal = PoseStamped()
    goal.header.frame_id = frame_id
    goal.header.stamp = rospy.Time.now()
    goal.pose.position.x = x
    goal.pose.position.y = y
    qx, qy, qz, qw = quaternion_from_euler(0.0, 0.0, math.radians(yaw_deg))
    goal.pose.orientation.x = qx
    goal.pose.orientation.y = qy
    goal.pose.orientation.z = qz
    goal.pose.orientation.w = qw
    return goal


def main():
    rospy.init_node('waypoint_navigation')
    pub = rospy.Publisher('/jetauto_1/move_base_simple/goal', PoseStamped, queue_size=1)
    rospy.sleep(2.0)

    for i, (x, y, yaw) in enumerate(WAYPOINTS, start=1):
        input(f'Press Enter to send waypoint {i}: x={x}, y={y}, yaw={yaw} deg')
        pub.publish(make_goal(x, y, yaw))
        rospy.loginfo('Published waypoint %d', i)


if __name__ == '__main__':
    main()

```


## Lab tasks
- Navigate to at least one RViz goal manually.
- Run the waypoint script for at least two poses.
- Record what happens when localization starts with an incorrect pose.
- Explain how global and local planning behave differently.

## Troubleshooting
- No path appears: check map, initial pose, costmap, and goal frame.
- Robot spins or drifts: check localization and transforms.
- Robot refuses to move: verify controller, emergency stop, app service, and topic names.
